# 🛡️ 02: Fraud Model Eval

Evaluation of the unsupervised `IsolationForest` scoring and deterministic rule booster.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

sys.path.append(os.path.dirname(os.getcwd()))

from components.A_fraud_detection.model import FraudModel
from components.A_fraud_detection.features import RealTimeFeatureEngine
from data.loader import load_transactions

df = load_transactions()
model = FraudModel()
engine = RealTimeFeatureEngine()

# Pre-fit on a slice
train_df = df.iloc[:2000]
test_df = df.iloc[2000:4000]

train_features = []
for _, row in train_df.iterrows():
    f, v = engine.compute_features(row.to_dict())
    train_features.append(v)

model.fit(np.array(train_features))
print("Model fitted on training slice.")

## 1. Anomaly Score Distribution
Visualizing the separation between 'normal' and 'anomalous' transactions.

In [ ]:
test_scores = []
flags = []
for _, row in test_df.iterrows():
    enriched, v = engine.compute_features(row.to_dict())
    scored = model.score(enriched, v)
    test_scores.append(scored['risk_score'])
    flags.append(scored['is_flagged'])

eval_df = pd.DataFrame({'score': test_scores, 'is_flagged': flags})

plt.figure(figsize=(10, 5))
sns.histplot(data=eval_df, x='score', hue='is_flagged', bins=50, element='step')
plt.title("Fraud Risk Score Distribution (Flagged vs Unflagged)")
plt.show()

## 2. Trigger Reasons
What's causing the flags?

In [ ]:
flagged_txns = eval_df[eval_df['is_flagged']]
print(f"Flagged {len(flagged_txns)} transactions out of {len(test_df)}.")